# 02 · Vietnamese ASR with faster-whisper

Produces `derived/asr/asr.parquet`.

High value for this corpus: the videos are TV news, so the narration frequently
states outright what a query is describing. Segment times are converted to frame
numbers here, because the search layer joins ASR spans onto catalog frames by
`frame_idx`.

In [ ]:
# --- Colab setup -------------------------------------------------------------
# Runtime > Change runtime type > T4 GPU before running.
!nvidia-smi -L
!git clone -q https://github.com/YOUR_ORG/new_aic2026.git /content/aic || (cd /content/aic && git pull -q)
%cd /content/aic
!pip install -q pandas pyarrow pillow tqdm
import sys; sys.path.insert(0, "/content/aic/src")

In [ ]:
# --- Mount Drive -------------------------------------------------------------
# Keyframes go in, artifacts come out. Keeping both on Drive means an interrupted
# runtime resumes instead of restarting from zero.
from google.colab import drive

drive.mount('/content/drive')

from pathlib import Path

DATA = Path('/content/drive/MyDrive/aic2026')
KEYFRAMES = DATA / 'raw/keyframes'
DERIVED   = DATA / 'derived'
DERIVED.mkdir(parents=True, exist_ok=True)
print('keyframes:', KEYFRAMES, KEYFRAMES.exists())

In [ ]:
!pip install -q faster-whisper

from faster_whisper import WhisperModel

model = WhisperModel("large-v3", device="cuda", compute_type="float16")

In [ ]:
import json
import subprocess
from pathlib import Path

VIDEOS = DATA / 'raw/videos'

def video_fps(path):
    """Frame rate, needed to convert segment times into submittable frame numbers."""
    out = subprocess.run(
        ['ffprobe', '-v', 'error', '-select_streams', 'v:0',
         '-show_entries', 'stream=r_frame_rate', '-of', 'json', str(path)],
        capture_output=True, text=True).stdout
    rate = json.loads(out)['streams'][0]['r_frame_rate']
    num, _, den = rate.partition('/')
    return float(num) / float(den or 1)

In [ ]:
import pandas as pd
from tqdm.auto import tqdm

OUT = DERIVED / 'asr'; OUT.mkdir(parents=True, exist_ok=True)
videos = sorted(VIDEOS.glob('*.mp4'))
rows = []

for video in tqdm(videos):
    shard = OUT / f'{video.stem}.parquet'
    if shard.exists():           # resume
        rows.append(pd.read_parquet(shard)); continue

    fps = video_fps(video)
    segments, _ = model.transcribe(str(video), language='vi', vad_filter=True)
    records = [{
        'video_id':   video.stem,
        'text':       s.text.strip(),
        'start_time': s.start,
        'end_time':   s.end,
        'start_frame': int(s.start * fps),
        'end_frame':   int(s.end * fps),
    } for s in segments if s.text.strip()]

    table = pd.DataFrame(records)
    table.to_parquet(shard, index=False)
    rows.append(table)

asr = pd.concat(rows, ignore_index=True)
asr.to_parquet(OUT / 'asr.parquet', index=False)
print(f'{len(asr):,} segments')
asr.head()

## Download

Copy to `data/derived/asr/asr.parquet`, then `aic build-text`.